# Rank Feature Lab

Sandbox for FeatureVector design. Scoring is the production algorithm:
StandardScaler + max cosine to Seeds (`rank_dayhikes` / `rank_scaled_rows`).
Do not train a ranker here.

Geometry and raw amenity points live in GeometryStore
(`{catalog}.geometry.sqlite`), not in Catalog. Production AmenityFacts stay
Trailhead-disk counts of hut/parking/restaurant_cafe/drinking_water.

Open `data/catalog.sqlite` plus `data/catalog.geometry.sqlite` when both exist
(re-ingest after this change). Otherwise build the committed fixture.

DEM / SolarAspect: `dayhike fetch-dem` writes FABDEM GeoTIFFs into `data/`
(gitignored). `dayhike sample-dem` copies those heights onto GeometryStore
vertices and Catalog elevation. This notebook still samples rasters for
SolarAspect; without a covering file it prints `dem skipped` and ranks.


In [ ]:
import json
import os
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
from sklearn.neighbors import BallTree

from dayhike import (
    Catalog,
    build_catalog,
    feature_matrix,
    fit_scaler_from_matrix,
    geometry_store_path_for,
    rank_dayhikes,
    rank_scaled_rows,
    scale_matrix,
)
from dayhike.catalog import EARTH_RADIUS_M, FEATURE_NAMES

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    if (candidate / "tests" / "fixtures" / "catalog.osm").is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
local_geometry = geometry_store_path_for(local)
if (
    local.is_file()
    and local_geometry.is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
):
    catalog_path = local
    geometry_path = local_geometry
    catalog = Catalog.open(catalog_path)
else:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    catalog = build_catalog(repo / "tests" / "fixtures" / "catalog.osm", catalog_path)
    geometry_path = geometry_store_path_for(catalog_path)

geo = sqlite3.connect(geometry_path)
print("catalog", catalog_path)
print("geometry_store", geometry_path)
print(catalog.ingest_report)
print("name_origin")
for origin, n in geo.execute(
    "SELECT name_origin, COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 1"
):
    print(f"  {origin}\t{n}")


## Production Ranking

`rank_dayhikes` is the real algorithm. Filter `name_origin != 'synthetic'` in
postprocessing if you only want OSM names.


In [ ]:
seed = next(
    hike.name for hike in catalog.dayhikes if hike.name_origin == "osm_name"
)
print("seed", seed)
production = rank_dayhikes(catalog, [seed])
for item in production.items:
    print(
        f"prod_rank\t{item.dayhike.id}\t{item.dayhike.name}\t"
        f"{item.dayhike.name_origin}\t{item.score:.6f}"
    )

named = [hike for hike in catalog.dayhikes if hike.name_origin == "osm_name"]
print("osm_name_only", len(named), "of", len(catalog.dayhikes))


## GeometryStore

Vertices, member-way tags, relation tags, and raw amenity points (including
Viewpoint, Peak, Shelter) with OSM tags as JSON.


In [ ]:
print("geometry tables")
for (name,) in geo.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1"
):
    count = geo.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
    print(f"  {name}\t{count}")
print("amenity kinds")
for kind, n in geo.execute(
    "SELECT kind, COUNT(*) FROM amenities GROUP BY 1 ORDER BY 1"
):
    print(f"  {kind}\t{n}")
sample = geo.execute(
    "SELECT osm_key, kind, tags FROM amenities WHERE json_extract(tags, '$.name') IS NOT NULL LIMIT 3"
).fetchall()
for osm_key, kind, tags in sample:
    print("tagged", osm_key, kind, json.loads(tags).get("name"))


## Map DayHikeGeometry

Plot assembled vertices. Under pytest this uses the Agg backend.


In [ ]:
import os

import geopandas
from shapely.geometry import LineString, Point

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")
import matplotlib.pyplot as plt

vertex_rows = list(
    geo.execute(
        "SELECT dayhike_id, seq, latitude, longitude FROM vertices ORDER BY dayhike_id, seq"
    )
)
by_hike: dict[int, list] = {}
for dayhike_id, _seq, lat, lon in vertex_rows:
    by_hike.setdefault(dayhike_id, []).append((lon, lat))
lines = [
    {"id": dayhike_id, "geometry": LineString(coords)}
    for dayhike_id, coords in by_hike.items()
    if len(coords) >= 2
]
gdf = geopandas.GeoDataFrame(lines, crs="EPSG:4326")
amenity_gdf = geopandas.GeoDataFrame(
    [
        {"kind": kind, "geometry": Point(lon, lat)}
        for kind, lat, lon in geo.execute(
            "SELECT kind, latitude, longitude FROM amenities"
        )
    ],
    crs="EPSG:4326",
)
fig, ax = plt.subplots(figsize=(6, 6))
gdf.plot(ax=ax, color="#2a6", linewidth=2)
if not amenity_gdf.empty:
    amenity_gdf.plot(ax=ax, column="kind", markersize=20, legend=True)
ax.set_title("DayHikeGeometry + amenities")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
try:
    from IPython.display import display

    display(fig)
except ImportError:
    pass
plt.close(fig)
print("mapped_dayhikes", len(gdf), "amenities", len(amenity_gdf))


## Experimental FeatureVector

Same scaler + max cosine, new columns: vertex count and hut count within 500 m
of DayHikeGeometry (corridor), not the Trailhead disk. Compare to production
Ranking on the same Seed.


In [ ]:
ids = [hike.id for hike in catalog.dayhikes]
names = [hike.name for hike in catalog.dayhikes]
by_id = {hike.id: hike for hike in catalog.dayhikes}
base = feature_matrix(catalog.dayhikes)
vertex_count = np.array(
    [
        geo.execute(
            "SELECT COUNT(*) FROM vertices WHERE dayhike_id = ?", (hike_id,)
        ).fetchone()[0]
        for hike_id in ids
    ],
    dtype=float,
)
hut_points = [
    (lat, lon)
    for lat, lon in geo.execute(
        "SELECT latitude, longitude FROM amenities WHERE kind = 'hut'"
    )
]
corridor_huts = np.zeros(len(ids), dtype=float)
if hut_points:
    tree = BallTree(np.radians(np.asarray(hut_points)), metric="haversine")
    radius = 500.0 / EARTH_RADIUS_M
    for index, hike_id in enumerate(ids):
        verts = list(
            geo.execute(
                "SELECT latitude, longitude FROM vertices WHERE dayhike_id = ? ORDER BY seq",
                (hike_id,),
            )
        )
        if not verts:
            continue
        hits = tree.query_radius(np.radians(np.asarray(verts)), r=radius)
        found: set[int] = set()
        for neighbours in hits:
            found.update(int(i) for i in neighbours)
        corridor_huts[index] = float(len(found))

exp_names = FEATURE_NAMES + ("vertex_count", "corridor_hut_count")
exp_matrix = np.column_stack([base, vertex_count, corridor_huts])
exp_scaler = fit_scaler_from_matrix(exp_matrix, exp_names)
seed_hike = by_id[next(hike.id for hike in catalog.dayhikes if hike.name == seed)]
exp_ranking = rank_scaled_rows(
    ids, names, scale_matrix(exp_matrix, exp_scaler), {seed_hike.id}
)
for dayhike_id, name, score in exp_ranking:
    print(f"exp_rank\t{dayhike_id}\t{name}\t{score:.6f}")

prod_order = [item.dayhike.id for item in production.items]
exp_order = [dayhike_id for dayhike_id, _name, _score in exp_ranking]
print("order_changed", prod_order != exp_order)


## DEM (FABDEM)

`dayhike fetch-dem` then `dayhike sample-dem` persist FABDEM heights onto
GeometryStore vertices and Catalog elevation min/max/gain. Sampling here is
only for SolarAspect inspection. Without a covering GeoTIFF this cell prints
`dem skipped`.


In [ ]:
from dayhike.dem import sample_elevations

points = [
    (lat, lon)
    for _id, lat, lon in geo.execute(
        "SELECT dayhike_id, latitude, longitude FROM vertices LIMIT 20"
    )
]
samples = sample_elevations(repo / "data", points)
finite = [z for z in samples if z is not None]
if not finite:
    print("dem skipped: no covering FABDEM tile in data/")
else:
    print("dem sampled", "n", len(finite), "min", min(finite), "max", max(finite))
    print("solar aspect is DEM-derived; not live weather")
geo.close()
